# Differential Expression Analysis with scutils

This notebook demonstrates a complete pseudobulk differential expression (DE)
workflow using `scutils` and `pyDESeq2`.

**Steps:**
1. Load / create a single-cell dataset with raw counts
2. Aggregate to pseudobulk with `scutils.pp.pseudobulk()`
3. Run DESeq2 with `scutils.tl.deseq2()`
4. Format results and visualise with `scutils.pl.volcano_plot()`

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

import scutils

print(f"scutils version: {scutils.__version__}")
print(f"scanpy version:  {sc.__version__}")

## 1. Create a synthetic dataset

We simulate a small dataset with:
- **6 donors** (biological replicates)
- **2 conditions**: `control` and `treated`
- **200 genes**, of which the first 20 are upregulated in `treated`
- Raw Poisson counts (as required by DESeq2)

In [ ]:
from anndata import AnnData
from scipy.sparse import csr_matrix

rng = np.random.default_rng(42)

n_genes = 200
donors = ["D1", "D2", "D3", "D4", "D5", "D6"]
conditions = ["control", "treated"]
cell_types = ["T_cell", "B_cell", "Monocyte"]
n_cells_per_group = 80

records = []
x_rows = []

for donor in donors:
    for cond in conditions:
        for ct in cell_types:
            for _ in range(n_cells_per_group):
                row = rng.poisson(30, size=n_genes)
                # Inject upregulation in genes 0-19 for treated
                if cond == "treated":
                    row[:20] = rng.poisson(90, size=20)
                x_rows.append(row)
                records.append(
                    {"donor": donor, "condition": cond, "cell_type": ct}
                )

obs = pd.DataFrame(records)
for col in ["donor", "condition", "cell_type"]:
    obs[col] = pd.Categorical(obs[col])
obs.index = [f"cell_{i}" for i in range(len(obs))]

var = pd.DataFrame(index=[f"gene_{i}" for i in range(n_genes)])

adata = AnnData(
    X=csr_matrix(np.vstack(x_rows).astype(np.float32)),
    obs=obs,
    var=var,
)

print(adata)
print(f"\nConditions: {adata.obs['condition'].cat.categories.tolist()}")
print(f"Donors:     {adata.obs['donor'].cat.categories.tolist()}")
print(f"Cell types: {adata.obs['cell_type'].cat.categories.tolist()}")

## 2. Pseudobulk aggregation

We aggregate cells per **donor × condition** combination. This is the
standard approach for comparing conditions across biological replicates.

Key parameters:
- `sample_col`: the column identifying biological replicates
- `groups_col`: an optional second column to stratify by (e.g. condition)
- `min_cells`: minimum cells required per group (groups below this are dropped)

In [ ]:
pb = scutils.pp.pseudobulk(
    adata,
    sample_col="donor",
    groups_col="condition",
    min_cells=10,
)

print(pb)
print(f"\nPseudobulk observations: {pb.n_obs}")
print(f"Genes:                   {pb.n_vars}")
pb.obs

## 3. Run DESeq2

`scutils.tl.deseq2()` wraps pyDESeq2's `DeseqDataSet` and `DeseqStats`.

Key parameters:
- `design`: a formulaic-style design formula (e.g. `"~condition"`)
- `contrast`: a 3-element list `[factor, test_level, reference_level]`
- `shrink_lfc`: apply empirical-Bayes LFC shrinkage (recommended for ranking)

In [ ]:
results = scutils.tl.deseq2(
    pb,
    design="~condition",
    contrast=["condition", "treated", "control"],
    shrink_lfc=True,
    alpha=0.05,
)

print(f"Results shape: {results.shape}")
print(f"Significant genes (padj < 0.05): {(results['padj'] < 0.05).sum()}")
results.sort_values("padj").head(10)

## 4. Volcano plot

The existing `scutils.pl.volcano_plot()` accepts any DataFrame with p-value
and log-fold-change columns.

### Option A: pass pyDESeq2 column names directly

In [ ]:
fig = scutils.pl.volcano_plot(
    results,
    pval_col="padj",
    lfc_col="log2FoldChange",
    pval_cutoff=0.05,
    lfc_cutoff=0.5,
    top_n_up=5,
    top_n_down=5,
    figsize=(10, 7),
)
plt.show()

### Option B: convert to Scanpy convention first

`scutils.tl.format_deseq2_results()` renames columns to the Scanpy
`rank_genes_groups` convention (`names`, `pvals_adj`, `logfoldchanges`),
so the volcano plot works with default parameters.

In [ ]:
df_scanpy = scutils.tl.format_deseq2_results(results)

fig = scutils.pl.volcano_plot(
    df_scanpy,
    pval_cutoff=0.05,
    lfc_cutoff=0.5,
    figsize=(10, 7),
)
plt.show()

## 5. Per-cell-type pseudobulk DE

A common use case is to run DE **per cell type** to find cell-type-specific
markers. Here we filter to one cell type, pseudobulk, and run DE.

In [ ]:
# Filter to T cells only
adata_tcell = adata[adata.obs["cell_type"] == "T_cell"].copy()
print(f"T cells: {adata_tcell.n_obs}")

# Pseudobulk per donor × condition
pb_tcell = scutils.pp.pseudobulk(
    adata_tcell,
    sample_col="donor",
    groups_col="condition",
    min_cells=10,
)
print(f"Pseudobulk observations: {pb_tcell.n_obs}")

# Run DESeq2
results_tcell = scutils.tl.deseq2(
    pb_tcell,
    design="~condition",
    contrast=["condition", "treated", "control"],
    shrink_lfc=True,
)

print(f"Significant genes (padj < 0.05): {(results_tcell['padj'] < 0.05).sum()}")

# Volcano plot
fig = scutils.pl.volcano_plot(
    results_tcell,
    pval_col="padj",
    lfc_col="log2FoldChange",
    pval_cutoff=0.05,
    lfc_cutoff=0.5,
    figsize=(10, 7),
)
fig.axes[0].set_title("T cells: treated vs control")
plt.show()

## Summary

| Step | Function | Description |
|------|----------|-------------|
| Pseudobulk | `scutils.pp.pseudobulk()` | Aggregate cells → one observation per sample |
| DE analysis | `scutils.tl.deseq2()` | Fit DESeq2 model and return results |
| Format results | `scutils.tl.format_deseq2_results()` | Rename columns to Scanpy convention |
| Visualise | `scutils.pl.volcano_plot()` | Volcano plot with gene annotations |